# Data Preprocessing Pipeline

**Dataset:** GoodScents + Leffingwell 

**Feature families:**
- MACCS keys: 166 binary bits
- Morgan fingerprints: 512 binary bits (radius=2)
- Mordred descriptors: 327 continuous physicochemical descriptors 

## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from skmultilearn.model_selection import iterative_train_test_split


## Load Data

In [2]:
df = pd.read_csv('goodscents_jadbio_ready.csv', sep=';')

LABEL_COLS = [
    'floral', 'fruity', 'sweet', 'woody', 'green', 'spicy',
    'animal_musk', 'earthy', 'citrus', 'chemical', 'gourmand', 'powdery_amber'
]

#Kanfr9o fingerprints 3la mordred descriptors 7it antraitiwhoum in different ways 

fp_cols      = [c for c in df.columns if c.startswith('MACCS_') or c.startswith('morgan_')]
mordred_cols = [c for c in df.columns if c not in LABEL_COLS + ['SMILES'] + fp_cols]

print(f'Total molecules      : {len(df)}')
print(f'MACCS + Morgan cols  : {len(fp_cols)}')
print(f'Mordred cols         : {len(mordred_cols)}')
print(f'Label cols           : {len(LABEL_COLS)}')

FileNotFoundError: [Errno 2] No such file or directory: 'goodscents_jadbio_ready.csv'

## Inspect NaN Values

The goal is to remove all  the columns where there's at least one  NAN value but here  when Inspected the the df i found that the the rows that has nan values are 5 for 327 mordred which is the mordred descriptors so instead of dropping the colums i dropped the rows 

In [ ]:
X_all = df[fp_cols + mordred_cols].values.astype(float)

#X7al mn row ou column fiha NaNs kan7sbouha bsum dyal boolean mask li kaydir True ila kayn NaN, w False ila ma kaynch, w kan7sbouha bsum bash n3rfou ch7al mn row ou column fiha NaNs

nan_rows = np.isnan(X_all).any(axis=1).sum()
nan_cols = np.isnan(X_all).any(axis=0).sum()

nan_cols_fp      = np.isnan(df[fp_cols].values.astype(float)).any(axis=0).sum()
nan_cols_mordred = np.isnan(df[mordred_cols].values.astype(float)).any(axis=0).sum()

nan_row_indices = list(np.where(np.isnan(X_all).any(axis=1))[0])

print(f'Rows with NaN        : {nan_rows}')
print(f'Columns with NaN     : {nan_cols}')
print(f'  - in fingerprints  : {nan_cols_fp}')
print(f'  - in Mordred       : {nan_cols_mordred}')

print()
print('SMILES of problematic molecules:')
for idx in nan_row_indices:
    print(f'  row {idx}: {df.iloc[idx]["SMILES"]}')

Rows with NaN        : 5
Columns with NaN     : 327
  - in fingerprints  : 0
  - in Mordred       : 327

SMILES of problematic molecules:
  row 3917: CC(C)=CCCC(C)CC(OCCCCCCCCCCC(C)C)OCCCCCCCCCCC(C)C
  row 3943: CC(C)CCCCCCCCCCOC(CC(C)CCCC(C)(C)O)OCCCCCCCCCCC(C)C
  row 3977: CC1=C2CC(C=O)C(C1)C(C(C)C)C2
  row 3978: CC(=O)C1CC2=C(C)CC1C(C(C)C)C2
  row 4646: CC(C)CCCCCCCCCCCCCCC(=O)OCC(O)COCC(COC(=O)CCCCCCCCCCCCCCC(C)C)OC(=O)CCCCCCCCCCCCCCC(C)C


## Drop NaN Rows


In [ ]:
df_clean = df.dropna().reset_index(drop=True)


print(f'Molecules before : {len(df)}')
print(f'Molecules after  : {len(df_clean)}')
print(f'Dropped          : {len(df) - len(df_clean)}')
print(f'NaN remaining    : {df_clean[fp_cols + mordred_cols].isna().sum().sum()}')

Molecules before : 4981
Molecules after  : 4976
Dropped          : 5
NaN remaining    : 0


## Split into Feature Groups

In [ ]:
X_fp      = df_clean[fp_cols].values.astype(float)
X_mordred = df_clean[mordred_cols].values.astype(float)
y         = df_clean[LABEL_COLS].values

print(f'X_fingerprints shape : {X_fp.shape}')
print(f'X_mordred shape      : {X_mordred.shape}')
print(f'y shape              : {y.shape}')

X_fingerprints shape : (4976, 678)
X_mordred shape      : (4976, 327)
y shape              : (4976, 12)


## Remove Zero-Variance Features

Features with zero variance are identical across all molecules  so , they carry no information and should be removed.
This is applied independently to each feature group.


In [ ]:
# Fingerprints
vt_fp = VarianceThreshold(threshold=0)
#kayselecti lcolumns li 3andhom variance > 0, w kaydroppihoum li 3andhom variance = 0 
X_fp = vt_fp.fit_transform(X_fp)
fp_cols_kept = np.array(fp_cols)[vt_fp.get_support()]

print(f'Fingerprints: {len(fp_cols)} -> {X_fp.shape[1]} features '
      f'({len(fp_cols) - X_fp.shape[1]} zero-variance dropped)')

# Mordred
vt_mordred = VarianceThreshold(threshold=0)
X_mordred = vt_mordred.fit_transform(X_mordred)
mordred_cols_kept = np.array(mordred_cols)[vt_mordred.get_support()]

print(f'Mordred     : {len(mordred_cols)} -> {X_mordred.shape[1]} features '
      f'({len(mordred_cols) - X_mordred.shape[1]} zero-variance dropped)')
print(f'\nTotal features after zero-variance removal: {X_fp.shape[1] + X_mordred.shape[1]}')

Fingerprints: 678 -> 672 features (6 zero-variance dropped)
Mordred     : 327 -> 327 features (0 zero-variance dropped)

Total features after zero-variance removal: 999


## Train/Test Split

I used iterative_train_test_split from scikit-multilearn cuz  For multi-label data with imbalanced labels, this  preserves the positive/negative ratio of each label independently in both train and test sets — critical given our class imbalance.

In [ ]:

# I concatenated  temporarily just for the split, then separate again
X_combined = np.hstack([X_fp, X_mordred])
n_fp = X_fp.shape[1]

X_train_comb, y_train, X_test_comb, y_test = iterative_train_test_split(
    X_combined, y, test_size=0.2
)

# Separate back into fingerprints and Mordred
X_fp_train    = X_train_comb[:, :n_fp]
X_mordred_train = X_train_comb[:, n_fp:]
X_fp_test     = X_test_comb[:, :n_fp]
X_mordred_test  = X_test_comb[:, n_fp:]

print(f'Train : {X_fp_train.shape[0]} molecules')
print(f'Test  : {X_fp_test.shape[0]} molecules')
print()

print(f'{"Label":<20} {"Full":>8} {"Train":>8} {"Test":>8}')
print('-' * 48)
for i, l in enumerate(LABEL_COLS):
    full  = y[:, i].mean() * 100
    train = y_train[:, i].mean() * 100
    test  = y_test[:, i].mean() * 100
    print(f'{l:<20} {full:>7.1f}% {train:>7.1f}% {test:>7.1f}%')

Train : 3877 molecules
Test  : 1099 molecules

Label                    Full    Train     Test
------------------------------------------------
floral                  26.2%    26.9%    23.7%
fruity                  45.8%    47.1%    41.5%
sweet                   38.7%    39.8%    35.0%
woody                   21.2%    21.7%    19.2%
green                   45.2%    46.4%    40.9%
spicy                   21.6%    22.1%    19.6%
animal_musk             15.5%    15.9%    14.0%
earthy                  20.2%    20.7%    18.3%
citrus                  10.3%    10.6%     9.1%
chemical                43.1%    44.2%    39.0%
gourmand                24.2%    25.0%    21.7%
powdery_amber           20.6%    21.1%    18.7%


## Scale Mordred Features



In [ ]:
scaler = StandardScaler()

# Fit on train, transform both
X_mordred_train = scaler.fit_transform(X_mordred_train)
X_mordred_test  = scaler.transform(X_mordred_test)      # same scaler, no refit

print('StandardScaler fitted on train only.')
print(f'Mordred train mean : {X_mordred_train.mean():.4f}')
print(f'Mordred train std  : {X_mordred_train.std():.4f}')
print(f'Mordred test mean  : {X_mordred_test.mean():.4f}')

StandardScaler fitted on train only.
Mordred train mean : -0.0000
Mordred train std  : 0.9985
Mordred test mean  : -0.0230


## Correlation Filter on Mordred (Train Only)

Highly correlated Mordred features (r > 0.95) are redundant ,  they encode the same information.
For each correlated pair, we drop one feature.

The correlation structure is computed on train only and the same column mask is applied to test.

In [ ]:
# Compute correlation matrix on train
corr_matrix = np.corrcoef(X_mordred_train.T)
corr_matrix = np.abs(corr_matrix)

# Find columns to drop
upper_triangle = np.triu(corr_matrix, k=1)
cols_to_drop = set()
rows, cols = np.where(upper_triangle > 0.95)
for r, c in zip(rows, cols):
    if c not in cols_to_drop:
        cols_to_drop.add(c)

cols_to_keep = [i for i in range(X_mordred_train.shape[1]) if i not in cols_to_drop]

# Apply same mask to both train and test
X_mordred_train = X_mordred_train[:, cols_to_keep]
X_mordred_test  = X_mordred_test[:, cols_to_keep]

print(f'Mordred features before correlation filter : {len(cols_to_keep) + len(cols_to_drop)}')
print(f'Features dropped (r > 0.95)               : {len(cols_to_drop)}')
print(f'Mordred features remaining                : {X_mordred_train.shape[1]}')

Mordred features before correlation filter : 327
Features dropped (r > 0.95)               : 6
Mordred features remaining                : 321


## Get the final feature Matrices



In [ ]:
X_train = np.hstack([X_fp_train, X_mordred_train])
X_test  = np.hstack([X_fp_test,  X_mordred_test])

print('Final feature matrices:')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')
print()


Final feature matrices:
  X_train : (3877, 999)
  X_test  : (1099, 999)
  y_train : (3877, 12)
  y_test  : (1099, 12)




# Model: Binary Relevance + Logistic Regression



## Build Classifier

In [ ]:
from skmultilearn.problem_transform import BinaryRelevance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score

# solver='saga' 7it 3andna large  feature spaces
clf = BinaryRelevance(
    classifier=LogisticRegression(
        class_weight='balanced',
        solver='saga',
        max_iter=300
    ),
    #lwla Convert features to dense NumPy array
    #tania Convert labels to dense NumPy array
    require_dense=[True, True]
)

print('Classifier:')
print(clf)

Classifier:
BinaryRelevance(classifier=LogisticRegression(class_weight='balanced',
                                              max_iter=300, solver='saga'),
                require_dense=[True, True])


## Hyperparameter Tuning with GridSearchCV

GridSearchCV finds the best regularization strength C by training BR+LR with each candidate C value and evaluating via 3-fold cross-validation using macro-F1.


In [14]:
scorer = make_scorer(f1_score, average='macro')

param_grid = {'classifier__C': [0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(
    clf,
    param_grid,
    scoring=scorer,
    cv=3,
    n_jobs=-1,
    verbose=1
)

print('Fitting GridSearchCV...')
grid_search.fit(X_train, y_train)

print(f'\nBest C         : {grid_search.best_params_["classifier__C"]}')
print(f'Best CV macro-F1 : {grid_search.best_score_:.3f}')

Fitting GridSearchCV...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Best C         : 0.01
Best CV macro-F1 : 0.553


## Predict on Test Set

scikit-multilearn returns sparse matrices by default so I  converted it  to dense numpy arrays for sklearn metrics.

In [17]:
best_clf = grid_search.best_estimator_

# Binary predictions (0/1) for F1
y_pred = np.array(best_clf.predict(X_test).todense())

# Probability scores for AUC
y_prob = np.array(best_clf.predict_proba(X_test).todense())

print(f'y_pred shape : {y_pred.shape}')
print(f'y_prob shape : {y_prob.shape}')

y_pred shape : (1110, 12)
y_prob shape : (1110, 12)


## Evaluate per Label

In [ ]:
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    balanced_accuracy_score, matthews_corrcoef,
    recall_score, confusion_matrix
)

rows = []

print(f'{"Label":<20} {"Bal.Acc":>8} {"MCC":>8} {"F1":>8} {"ROC AUC":>8} {"PR AUC":>8} {"Sensitivity":>12} {"Specificity":>12}')
print('-' * 96)

for i, label in enumerate(LABEL_COLS):
    yt = y_test[:, i]
    yp = y_pred[:, i]
    ypr = y_prob[:, i]

    bal_acc  = balanced_accuracy_score(yt, yp)
    mcc      = matthews_corrcoef(yt, yp)
    f1       = f1_score(yt, yp, zero_division=0)
    
    try:    roc_auc = roc_auc_score(yt, ypr)
    except: roc_auc = float('nan')
    
    try:    pr_auc = average_precision_score(yt, ypr)
    except: pr_auc = float('nan')

    sensitivity = recall_score(yt, yp, zero_division=0)  
    
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan') 

    rows.append({
        'Label'       : label,
        'Bal.Acc'     : round(bal_acc, 3),
        'MCC'         : round(mcc, 3),
        'F1'          : round(f1, 3),
        'ROC_AUC'     : round(roc_auc, 3),
        'PR_AUC'      : round(pr_auc, 3),
        'Sensitivity' : round(sensitivity, 3),
        'Specificity' : round(specificity, 3),
    })

    print(f'{label:<20} {bal_acc:>8.3f} {mcc:>8.3f} {f1:>8.3f} {roc_auc:>8.3f} {pr_auc:>8.3f} {sensitivity:>12.3f} {specificity:>12.3f}')

print('-' * 96)

# Macro averages
results = pd.DataFrame(rows)
macro = results.drop(columns='Label').mean()
print(f'{"MACRO AVG":<20} {macro["Bal.Acc"]:>8.3f} {macro["MCC"]:>8.3f} {macro["F1"]:>8.3f} {macro["ROC_AUC"]:>8.3f} {macro["PR_AUC"]:>8.3f} {macro["Sensitivity"]:>12.3f} {macro["Specificity"]:>12.3f}')

Label                 Bal.Acc      MCC       F1  ROC AUC   PR AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.812    0.546    0.656    0.875    0.672        0.854        0.771
fruity                  0.772    0.537    0.736    0.852    0.804        0.776        0.768
sweet                   0.682    0.348    0.609    0.742    0.595        0.784        0.579
woody                   0.771    0.454    0.564    0.837    0.567        0.763        0.779
green                   0.713    0.419    0.673    0.776    0.667        0.740        0.686
spicy                   0.737    0.380    0.507    0.805    0.505        0.795        0.678
animal_musk             0.703    0.302    0.406    0.797    0.442        0.662        0.743
earthy                  0.738    0.395    0.515    0.820    0.571        0.700        0.776
citrus                  0.766    0.354    0.393    0.851    0.390        0.

## Comparison Table 

In [ ]:


#  BR + LR results 
br_lr = {
    'floral'       : {'Bal.Acc':0.812,'MCC':0.546,'F1':0.656,'ROC_AUC':0.875,'PR_AUC':0.672,'Sensitivity':0.854,'Specificity':0.771},
    'fruity'       : {'Bal.Acc':0.772,'MCC':0.537,'F1':0.736,'ROC_AUC':0.852,'PR_AUC':0.804,'Sensitivity':0.776,'Specificity':0.768},
    'sweet'        : {'Bal.Acc':0.682,'MCC':0.348,'F1':0.609,'ROC_AUC':0.742,'PR_AUC':0.595,'Sensitivity':0.784,'Specificity':0.579},
    'woody'        : {'Bal.Acc':0.771,'MCC':0.454,'F1':0.564,'ROC_AUC':0.837,'PR_AUC':0.567,'Sensitivity':0.763,'Specificity':0.779},
    'green'        : {'Bal.Acc':0.713,'MCC':0.419,'F1':0.673,'ROC_AUC':0.776,'PR_AUC':0.667,'Sensitivity':0.740,'Specificity':0.686},
    'spicy'        : {'Bal.Acc':0.737,'MCC':0.380,'F1':0.507,'ROC_AUC':0.805,'PR_AUC':0.505,'Sensitivity':0.795,'Specificity':0.678},
    'animal_musk'  : {'Bal.Acc':0.703,'MCC':0.302,'F1':0.406,'ROC_AUC':0.797,'PR_AUC':0.442,'Sensitivity':0.662,'Specificity':0.743},
    'earthy'       : {'Bal.Acc':0.738,'MCC':0.395,'F1':0.515,'ROC_AUC':0.820,'PR_AUC':0.571,'Sensitivity':0.700,'Specificity':0.776},
    'citrus'       : {'Bal.Acc':0.766,'MCC':0.354,'F1':0.393,'ROC_AUC':0.851,'PR_AUC':0.390,'Sensitivity':0.735,'Specificity':0.797},
    'chemical'     : {'Bal.Acc':0.717,'MCC':0.423,'F1':0.668,'ROC_AUC':0.779,'PR_AUC':0.654,'Sensitivity':0.765,'Specificity':0.670},
    'gourmand'     : {'Bal.Acc':0.748,'MCC':0.419,'F1':0.557,'ROC_AUC':0.817,'PR_AUC':0.584,'Sensitivity':0.776,'Specificity':0.719},
    'powdery_amber': {'Bal.Acc':0.739,'MCC':0.393,'F1':0.515,'ROC_AUC':0.810,'PR_AUC':0.465,'Sensitivity':0.727,'Specificity':0.751},
}

# JadBio results 
jadbio = {
    'floral'       : {'Bal.Acc':0.750,'MCC':0.526,'F1':0.638,'ROC_AUC':0.857,'PR_AUC':0.817,'Sensitivity':0.595,'Specificity':0.905},
    'fruity'       : {'Bal.Acc':0.799,'MCC':0.600,'F1':0.778,'ROC_AUC':0.872,'PR_AUC':0.871,'Sensitivity':0.762,'Specificity':0.835},
    'sweet'        : {'Bal.Acc':0.681,'MCC':0.366,'F1':0.603,'ROC_AUC':0.753,'PR_AUC':0.749,'Sensitivity':0.587,'Specificity':0.775},
    'woody'        : {'Bal.Acc':0.691,'MCC':0.468,'F1':0.530,'ROC_AUC':0.845,'PR_AUC':0.800,'Sensitivity':0.427,'Specificity':0.955},
    'green'        : {'Bal.Acc':0.709,'MCC':0.426,'F1':0.665,'ROC_AUC':0.796,'PR_AUC':0.796,'Sensitivity':0.623,'Specificity':0.794},
    'spicy'        : {'Bal.Acc':0.629,'MCC':0.354,'F1':0.410,'ROC_AUC':0.794,'PR_AUC':0.740,'Sensitivity':0.302,'Specificity':0.956},
    'animal_musk'  : {'Bal.Acc':0.665,'MCC':0.449,'F1':0.472,'ROC_AUC':0.805,'PR_AUC':0.753,'Sensitivity':0.354,'Specificity':0.975},
    'earthy'       : {'Bal.Acc':0.683,'MCC':0.470,'F1':0.516,'ROC_AUC':0.802,'PR_AUC':0.776,'Sensitivity':0.401,'Specificity':0.964},
    'citrus'       : {'Bal.Acc':None, 'MCC':None, 'F1':None, 'ROC_AUC':None, 'PR_AUC':None, 'Sensitivity':None,'Specificity':None},
    'chemical'     : {'Bal.Acc':0.728,'MCC':0.466,'F1':0.678,'ROC_AUC':0.800,'PR_AUC':0.787,'Sensitivity':0.639,'Specificity':0.817},
    'gourmand'     : {'Bal.Acc':0.671,'MCC':0.450,'F1':0.502,'ROC_AUC':0.845,'PR_AUC':0.807,'Sensitivity':0.377,'Specificity':0.964},
    'powdery_amber': {'Bal.Acc':0.628,'MCC':0.317,'F1':0.403,'ROC_AUC':0.792,'PR_AUC':0.717,'Sensitivity':0.323,'Specificity':0.933},
}

#  Build comparison table 
metrics = ['Bal.Acc','MCC','F1','ROC_AUC','PR_AUC','Sensitivity','Specificity']
rows = []

for label in LABEL_COLS:
    row = {'Label': label}
    for m in metrics:
        row[f'BR_LR_{m}']  = br_lr[label][m]
        row[f'JadBio_{m}'] = jadbio[label][m]
    rows.append(row)

results = pd.DataFrame(rows)

# Macro averages 
macro_br    = {m: round(pd.DataFrame(br_lr).T[m].mean(), 3) for m in metrics}
macro_jadbio = {m: round(pd.DataFrame({k:v for k,v in jadbio.items() if v['F1'] is not None}).T[m].astype(float).mean(), 3) for m in metrics}

macro_row = {'Label': 'MACRO AVG'}
for m in metrics:
    macro_row[f'BR_LR_{m}']  = macro_br[m]
    macro_row[f'JadBio_{m}'] = macro_jadbio[m]

results = pd.concat([results, pd.DataFrame([macro_row])], ignore_index=True)

# Display
print('BR+LR vs JadBio — Full Metric Comparison')
print()
print(f'{"Label":<20}', end='')
for m in metrics:
    print(f'  {"BR_"+m:>12} {"JDB_"+m:>12}', end='')
print()
print('-' * (20 + 28*len(metrics)))
for _, row in results.iterrows():
    print(f'{row["Label"]:<20}', end='')
    for m in metrics:
        br_val  = row[f'BR_LR_{m}']
        jdb_val = row[f'JadBio_{m}']
        br_str  = f'{br_val:.3f}'  if pd.notna(br_val)  else ' None'
        jdb_str = f'{jdb_val:.3f}' if pd.notna(jdb_val) else ' None'
        print(f'  {br_str:>12} {jdb_str:>12}', end='')
    print()

BR+LR vs JadBio — Full Metric Comparison

Label                   BR_Bal.Acc  JDB_Bal.Acc        BR_MCC      JDB_MCC         BR_F1       JDB_F1    BR_ROC_AUC  JDB_ROC_AUC     BR_PR_AUC   JDB_PR_AUC  BR_Sensitivity JDB_Sensitivity  BR_Specificity JDB_Specificity
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
floral                       0.812        0.750         0.546        0.526         0.656        0.638         0.875        0.857         0.672        0.817         0.854        0.595         0.771        0.905
fruity                       0.772        0.799         0.537        0.600         0.736        0.778         0.852        0.872         0.804        0.871         0.776        0.762         0.768        0.835
sweet                        0.682        0.681         0.348        0.366         0.609        0.603

# Model: Binary Relevance + Random Forest

RF is tree-based so it's scale-invariant — no need to re-scale. We reuse X_train/X_test directly.
`class_weight='balanced'` handles imbalance at the tree level.
GridSearchCV tunes `n_estimators` and `max_features` using macro-F1.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from skmultilearn.problem_transform import BinaryRelevance
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

rf_clf = BinaryRelevance(
    classifier=RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    require_dense=[True, True]
)

print('Classifier:')
print(rf_clf)

## Hyperparameter Tuning — BR+RF

We search over `n_estimators` and `max_features`.
`max_features='sqrt'` is the RF default for classification; `'log2'` is more aggressive on high-dimensional fingerprint spaces.

In [ ]:
scorer = make_scorer(f1_score, average='macro')

param_grid_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_features': ['sqrt', 'log2'],
}

grid_search_rf = GridSearchCV(
    rf_clf,
    param_grid_rf,
    scoring=scorer,
    cv=3,
    n_jobs=-1,
    verbose=1
)

print('Fitting GridSearchCV for BR+RF...')
grid_search_rf.fit(X_train, y_train)

print(f'\nBest params      : {grid_search_rf.best_params_}')
print(f'Best CV macro-F1 : {grid_search_rf.best_score_:.3f}')

## Predict on Test Set

In [ ]:
best_rf_clf = grid_search_rf.best_estimator_

y_pred_rf = np.array(best_rf_clf.predict(X_test).todense())
y_prob_rf = np.array(best_rf_clf.predict_proba(X_test).todense())

print(f'y_pred shape : {y_pred_rf.shape}')
print(f'y_prob shape : {y_prob_rf.shape}')

## Evaluate per Label — BR+RF

In [ ]:
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    balanced_accuracy_score, matthews_corrcoef,
    recall_score, confusion_matrix
)

rows_rf = []

print(f'{"Label":<20} {"Bal.Acc":>8} {"MCC":>8} {"F1":>8} {"ROC AUC":>8} {"PR AUC":>8} {"Sensitivity":>12} {"Specificity":>12}')
print('-' * 96)

for i, label in enumerate(LABEL_COLS):
    yt  = y_test[:, i]
    yp  = y_pred_rf[:, i]
    ypr = y_prob_rf[:, i]

    bal_acc = balanced_accuracy_score(yt, yp)
    mcc     = matthews_corrcoef(yt, yp)
    f1      = f1_score(yt, yp, zero_division=0)

    try:    roc_auc = roc_auc_score(yt, ypr)
    except: roc_auc = float('nan')

    try:    pr_auc = average_precision_score(yt, ypr)
    except: pr_auc = float('nan')

    sensitivity = recall_score(yt, yp, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')

    rows_rf.append({
        'Label'       : label,
        'Bal.Acc'     : round(bal_acc, 3),
        'MCC'         : round(mcc, 3),
        'F1'          : round(f1, 3),
        'ROC_AUC'     : round(roc_auc, 3),
        'PR_AUC'      : round(pr_auc, 3),
        'Sensitivity' : round(sensitivity, 3),
        'Specificity' : round(specificity, 3),
    })

    print(f'{label:<20} {bal_acc:>8.3f} {mcc:>8.3f} {f1:>8.3f} {roc_auc:>8.3f} {pr_auc:>8.3f} {sensitivity:>12.3f} {specificity:>12.3f}')

print('-' * 96)
results_rf = pd.DataFrame(rows_rf)
macro_rf = results_rf.drop(columns='Label').mean()
print(f'{"MACRO AVG":<20} {macro_rf["Bal.Acc"]:>8.3f} {macro_rf["MCC"]:>8.3f} {macro_rf["F1"]:>8.3f} {macro_rf["ROC_AUC"]:>8.3f} {macro_rf["PR_AUC"]:>8.3f} {macro_rf["Sensitivity"]:>12.3f} {macro_rf["Specificity"]:>12.3f}')

## Three-way Comparison: BR+LR vs BR+RF vs JadBio

In [ ]:
# BR+RF results dict (populated from rows_rf above)
br_rf = {r['Label']: {k: v for k, v in r.items() if k != 'Label'} for r in rows_rf}

# Reuse br_lr and jadbio dicts from above
metrics = ['Bal.Acc', 'MCC', 'F1', 'ROC_AUC', 'PR_AUC', 'Sensitivity', 'Specificity']

print('BR+LR vs BR+RF vs JadBio — Full Metric Comparison')
print()
header = f'{"Label":<20}'
for m in metrics:
    header += f'  {"LR_"+m:>10} {"RF_"+m:>10} {"JDB_"+m:>10}'
print(header)
print('-' * (20 + 34 * len(metrics)))

for label in LABEL_COLS:
    row_str = f'{label:<20}'
    for m in metrics:
        lr_val  = br_lr[label][m]
        rf_val  = br_rf[label][m]
        jdb_val = jadbio[label][m]
        def fmt(v):
            if v is None: return ' None'
            try:
                if np.isnan(v): return ' None'
            except: pass
            return f'{v:.3f}'
        row_str += f'  {fmt(lr_val):>10} {fmt(rf_val):>10} {fmt(jdb_val):>10}'
    print(row_str)

print('-' * (20 + 34 * len(metrics)))
macro_row_str = f'{"MACRO AVG":<20}'
macro_lr_d  = {m: round(pd.DataFrame(br_lr).T[m].mean(), 3) for m in metrics}
macro_rf_d  = {m: round(results_rf[m].mean(), 3) for m in metrics}
macro_jdb_d = {m: round(pd.DataFrame({k:v for k,v in jadbio.items() if v['F1'] is not None}).T[m].astype(float).mean(), 3) for m in metrics}
for m in metrics:
    macro_row_str += f'  {macro_lr_d[m]:>10.3f} {macro_rf_d[m]:>10.3f} {macro_jdb_d[m]:>10.3f}'
print(macro_row_str)